# turkish_distill: Mizan evaluation (secondary benchmark)

Evaluates embedding models on the **Turkish tasks of the Mizan leaderboard** (14 tasks, [leaderboard](https://newmindai-mizan.hf.space)) with stock `mteb` 1.38.51. This is our **development benchmark**: every decision of the project was made by looking at TR-MTEB, which biases the final TR-MTEB number; Mizan shares only 5 of its 14 tasks with TR-MTEB, the other **9 independent tasks** (Massive intent and scenario, multilingual sentiment, SIB200 classification and clustering, WebFAQ and MKQA retrieval, XNLI v2, STS22) are what we pick models and data recipes by. TR-MTEB (`colab_eval.ipynb`) stays the final report.

- Results go to Drive: `bitirme/eval_mizan/<label>/` (one folder per model; the labels are the same as in `eval/`). An interrupted run resumes and skips finished tasks.
- Use a **separate runtime** from `colab_eval.ipynb`: it needs mteb 1.38.51 and `datasets<4`, the TR-MTEB fork pins mteb 1.31.3 and `datasets<3`.
- Protocol: plain encoding, no instruction or prefix, max length 512 (the leaderboard page does not document its own protocol, so absolute numbers may differ a little from the leaderboard).
- The training data was cleaned against the Mizan test texts too (`benchmark_texts` export of both lists), so these tasks stay held out.

Before the first run: `HF_TOKEN` in Colab Secrets.

## 1. Setup

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'
EVAL_DIR = f'{DRIVE_ROOT}/eval_mizan'

os.makedirs(EVAL_DIR, exist_ok=True)
# HF cache on the LOCAL disk (Drive corrupts large model files).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
# Mizan runs on mteb 1.38.51. That release asks for Python < 3.13 and datasets < 4; this runtime's Python may be newer
# (the package is pure Python, so --ignore-requires-python is enough) and MKQA is a loading script, which needs datasets < 4.
!pip install --quiet --ignore-requires-python "mteb==1.38.51" "datasets<4"
!git log -1 --oneline

Check that mteb 1.38.51 and all 14 Turkish tasks are available. If this fails, restart the runtime once, then **re-run the Setup cell above again** (a restart clears all variables such as `DRIVE_ROOT` and `EVAL_DIR`):

In [ ]:
from turkish_distill.eval.mizan import TASK_CATEGORIES, INDEPENDENT_TASKS, SHARED_WITH_TRMTEB, patch_transformers_for_mteb

# mteb 1.38 imports a transformers name that transformers 5 removed; this adds the alias (must run before `import mteb`)
print('transformers alias added:', patch_transformers_for_mteb())
import mteb

print('mteb', getattr(mteb, '__version__', '?'))
tasks = mteb.get_tasks(tasks=list(TASK_CATEGORIES), languages=['tur'])
print(len(tasks), 'Turkish tasks found')
print('independent (development set):', INDEPENDENT_TASKS)
print('shared with TR-MTEB (leaderboard only):', SHARED_WITH_TRMTEB)

## 2. Model presets

The same presets (same names and revisions) as in `colab_eval.ipynb`, so a model has the same label in `eval/` and `eval_mizan/`. Presets with instruction prefixes are dropped: this notebook evaluates plain encoding only. Add entries for new runs (`model` = the `final/` folder on Drive).

In [ ]:
INSTR = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

# Prompt-sensitive models are scored twice: plain (the TR-MTEB authors' protocol) and with their
# query/document prefixes (`*_instr`). Prompts only change RETRIEVAL scores (verified on mE5: the 15 other
# tasks are bit-identical), so every `*_instr` preset has `reuse_from=<its plain preset>`: it copies the
# plain run's non-retrieval results and only evaluates the 11 retrieval tasks. Run the plain preset first.
# Only the EmbeddingGemma prefixes below were checked against its model card; the Qwen3 instruction was
# written from memory. Models without an `_instr` preset are run plain only, so their retrieval score may
# understate what they reach with the prefixes they expect.
PRESETS = {
    # --- teacher candidates ---
    'bge_m3': dict(model='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181', dtype='float16'),
    'me5_large_instruct': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16'),
    'me5_large_instruct_instr': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16', query_prompt=INSTR,
                                     reuse_from='me5_large_instruct'),
    'qwen3_8b': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16),
    'qwen3_8b_instr': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_8b'),
    'turkish_e5_large': dict(model='ytu-ce-cosmos/turkish-e5-large', revision='02e2362d503bbdeafcb17143b2165c0743f9fdb1', dtype='float16'),

    # --- more teacher candidates (added 2026-10-11) ---
    # Gated: accept the license on the model page with the same Hugging Face account as HF_TOKEN.
    'embeddinggemma_300m': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16'),
    'embeddinggemma_300m_instr': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16',
                                      query_prompt='task: search result | query: ', passage_prompt='title: none | text: ',
                                      reuse_from='embeddinggemma_300m'),
    'harrier_0_6b': dict(model='microsoft/harrier-oss-v1-0.6b', revision='f9b9dc8d367d443f2479d27aa5d8d2850c0774ee', dtype='bfloat16'),
    'harrier_270m': dict(model='microsoft/harrier-oss-v1-270m', revision='31de22b673913c7d658c0f03f792d77c2dcf8ebd', dtype='bfloat16'),
    'gte_multilingual_base': dict(model='Alibaba-NLP/gte-multilingual-base', revision='9bbca17d9273fd0d03d5725c7a4b0f6b45142062', dtype='float16', trust_remote_code=True),
    'arctic_embed_l_v2': dict(model='Snowflake/snowflake-arctic-embed-l-v2.0', revision='ac6544c8a46e00af67e330e85a9028c66b8cfd9a', dtype='float16'),
    'nomic_embed_v2_moe': dict(model='nomic-ai/nomic-embed-text-v2-moe', revision='1066b6599d099fbb93dfcb64f9c37a7c9e503e85', dtype='float32', trust_remote_code=True),
    'qwen3_4b': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16),
    'qwen3_4b_instr': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_4b'),
    'qwen3_0_6b': dict(model='Qwen/Qwen3-Embedding-0.6B', revision='97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3', dtype='bfloat16'),

    # --- baselines from the same lab ---
    'modernbert_tr_embed': dict(model='ytu-ce-cosmos/modernbert-tr-embed', revision='855820e0079102805908d60d32cd9fd59c970b80'),
    'modernbert_tr_1k_embed': dict(model='ytu-ce-cosmos/modernbert-tr-1k-embed', revision='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    # --- our student before any distillation (raw MLM encoder + mean pooling) ---
    'modernbert_tr_base_1k_raw': dict(model='ytu-ce-cosmos/modernbert-tr-base-1k', revision='c2e35f553101cce5b03fbb318ef8728473fd34dd'),

    # --- our trained students (stage 1): `model` is the run's final/ folder on Drive. Same protocol as the raw
    # student above (plain encoding, mean pooling), so the numbers are directly comparable. Add one line per run.
    'c_mnrl_s20_b128': dict(model=f'{DRIVE_ROOT}/runs/c_modernbert-tr-base-1k_mnrl_s20_b128_chunk32/20261018-053836-3551f77/final'),
    'c_mnrl_s50_b128': dict(model=f'{DRIVE_ROOT}/runs/c_modernbert-tr-base-1k_mnrl_s50_b128_chunk32/20261022-074944-35c09db/final'),
    'c_mnrl_symmetric_s20_b128': dict(model=f'{DRIVE_ROOT}/runs/c_modernbert-tr-base-1k_mnrl_symmetric_s20_b128_chunk32/20261022-074948-35c09db/final'),
}
print(list(PRESETS))

In [ ]:
# Stage-1 distilled students: the final/ folder of each run on Drive (same plain protocol as every other preset).
# d_single_*: one teacher; d_two_*: both teachers, equal weights; d_two_bge*_qwen*: static weight ratios BGE:Qwen;
# d_two_qwenramp900: Qwen's weight rises from 0 to 1 over the first 900 of 1800 steps;
# d_single_me5 / d_two_bge_m3_me5: multilingual-e5-large-instruct alone / with BGE-M3 (equal weights).
# Evaluate only finished runs (final/ exists).
PRESETS.update({
    'd_single_qwen3_8b': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_qwen3_8b/20261018-094046-e346476/final'),
    'd_single_bge_m3': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_bge_m3/20261018-094050-e346476/final'),
    'd_two_qwen3_8b_bge_m3': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_qwen3_8b_bge_m3/20261018-094149-e346476/final'),
    'd_two_bge3_qwen1': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge3_qwen1/20261018-134441-4fde652/final'),
    'd_two_bge1_qwen3': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge1_qwen3/20261018-134500-4fde652/final'),
    'd_two_qwenramp900': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_qwenramp900/20261018-134505-4fde652/final'),
    'd_single_me5': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_me5/20261019-051608-852baf8/final'),
    'd_two_bge_m3_me5': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge_m3_me5/20261019-051622-852baf8/final'),
    'd_mixabl_cosmos70k': dict(model=f'{DRIVE_ROOT}/runs/mixabl_bge_cosmos70k/20261019-085142-6e5c138/final'),
    'd_mixabl_mix70k': dict(model=f'{DRIVE_ROOT}/runs/mixabl_bge_mix70k/20261019-085205-6e5c138/final'),
    'd_ablloss_relation_only': dict(model=f'{DRIVE_ROOT}/runs/ablloss_bge_relation_only/20261019-085218-6e5c138/final'),
    'd_single_bge_m3_seed43': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_bge_m3/20261021-141516-15d16ee/final'),
    'd_ablloss_vector_only': dict(model=f'{DRIVE_ROOT}/runs/ablloss_bge_vector_only/20261021-141407-15d16ee/final'),
    'd_single_bge_m3_seed44': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_bge_m3/20261022-051724-471f9e8/final'),
    'd_mixabl_mix70k_seed43': dict(model=f'{DRIVE_ROOT}/runs/mixabl_bge_mix70k/20261022-051746-471f9e8/final'),
    # equal weights (1:1): the weights in this run's config.yaml are 1.0 / 1.0, not the planned 3:1
    'd_two_qwen3_8b_bge_m3_seed43': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_qwen3_8b_bge_m3/20261022-051816-471f9e8/final'),
    'd_single_cosmos_e5': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_cosmos_e5/20261022-184342-ad8b834/final'),
    'd_single_me5_center': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_me5_center/20261022-184557-ad8b834/final'),
    # v1 = the first Gram standardisation (student z-scored too); the fixed version gets the plain name
    'd_single_me5_gram_v1': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_me5_gram/20261022-184358-ad8b834/final'),
    # fixed Gram standardisation: only the teacher's matrix is standardised to a reference spread, the student stays raw
    'd_single_me5_gram': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_me5_gram/20261023-073422-6d93917/final'),
    'd_single_modernbert_tr_embed': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_modernbert_tr_embed/20261023-074841-6d93917/final'),
    # METHOD='center' centred BOTH teachers (the config shows center: true twice), not only mE5
    'd_two_bge_m3_me5_center': dict(model=f'{DRIVE_ROOT}/runs/stage1_two_bge_m3_me5_center/20261023-073752-6d93917/final'),
    # the teacher encoded the question texts with its query instruction (passages plain); the student sees raw text
    'd_single_cosmos_e5_instr': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_cosmos_e5_instr/20261023-100107-f5b43e8/final'),
    'd_single_qwen3_8b_instr': dict(model=f'{DRIVE_ROOT}/runs/stage1_single_qwen3_8b_instr/20261023-100201-f5b43e8/final'),
    # scaling: the same recipe as d_single_bge_m3 / d_single_modernbert_tr_embed, but trained on the 200K mix (mix_v2_200k), same 1800 steps
    'd_scale_bge_mix200k': dict(model=f'{DRIVE_ROOT}/runs/scale_bge_mix200k/20261023-180254-95d9ef4/final'),
    'd_scale_8k_mix200k': dict(model=f'{DRIVE_ROOT}/runs/scale_8k_mix200k/20261023-180318-95d9ef4/final'),
    # same recipe and 200K mix as d_scale_bge_mix200k, but 3x the steps (5400 instead of 1800): is the budget the limit?
    'd_scale_bge_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_bge_mix200k_s5400/20261024-054957-f09f000/final'),
    # 8K teacher on the 500K mix (mix_v2_500k), same 1800 steps: 0.46 epochs
    'd_scale_8k_mix500k': dict(model=f'{DRIVE_ROOT}/runs/scale_8k_mix500k/20261024-094556-f09f000/final'),
    # 5400-step students on the 200K mix (8K teacher alone; plain and shared-relation BGE-M3 + 8K-embed pairs)
    'd_scale_8k_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_8k_mix200k_s5400/20261024-110104-dad457c/final'),
    'd_two_bge_8k_plain_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_two_bge_8k_mix200k_s5400/20261024-125800-dad457c/final'),
    'd_two_bge_8k_shared_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_two_bge_8k_mix200k_s5400_shared/20261024-133825-3c9b263/final'),
    # seed 43 of the two single-teacher 5400-step runs (seed 42 above), and the plain pair with the 8K base as the student
    'd_scale_bge_mix200k_s5400_seed43': dict(model=f'{DRIVE_ROOT}/runs/scale_bge_mix200k_s5400_seed43/20261025-072000-seed43/final'),
    'd_scale_8k_mix200k_s5400_seed43': dict(model=f'{DRIVE_ROOT}/runs/scale_8k_mix200k_s5400_seed43/20261025-080234-4e2bd4b/final'),
    'd_two_bge_8k_plain_s5400_base8k': dict(model=f'{DRIVE_ROOT}/runs/scale_two_bge_8k_mix200k_s5400_base8k/20261025-080532-4e2bd4b/final'),
    # round 2: a fresh 1k-base student distilled from the plain BGE-M3 + 8K-embed student (config stage2_single_two_teacher_student); and the same pair trained for 10800 steps
    'd_round2_from_two_s5400': dict(model=f'{DRIVE_ROOT}/runs/round2_from_two_s5400/20261025-141207-386f1c0/final'),
    'd_two_bge_8k_plain_s10800': dict(model=f'{DRIVE_ROOT}/runs/scale_two_bge_8k_mix200k_s10800/20261025-173323-d5aa802/final'),
    # Qwen3-8B on the 200K mix, 5400 steps, 1k base: alone, and as a third teacher next to BGE-M3 + 8K-embed (equal weights, and Qwen weight 0.5)
    'd_scale_qwen_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_qwen_mix200k_s5400/20261025-190627-19f1584/final'),
    'd_three_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_three_mix200k_s5400/20261025-191615-19f1584/final'),
    'd_three_mix200k_s5400_qwen05': dict(model=f'{DRIVE_ROOT}/runs/scale_three_mix200k_s5400_qwen05/20261025-190601-19f1584/final'),
    # Qwen vectors projected onto their top 768 principal components (teachers[].pca_dim=768): alone and as the third teacher; and the BGE-M3 + 8K-embed pair with the listwise neighbour term (weight 10)
    'd_scale_qwen_pca768_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_qwen_pca768_mix200k_s5400/20261026-053029-5843804/final'),
    'd_three_pca768_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_three_pca768_mix200k_s5400/20261026-053112-5843804/final'),
    'd_two_listwise10_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_two_listwise10_mix200k_s5400/20261026-055059-a0b6481/final'),
    # BGE-M3 + 8K-embed pair with neighbour-rich batches (train.neighbor_fraction=0.5) and the listwise term (weight 10)
    'd_two_nb_listwise10_mix200k_s5400': dict(model=f'{DRIVE_ROOT}/runs/scale_two_nb_listwise10_mix200k_s5400/20261026-125852-7df133f/final'),
    # weight averages (alpha 0.5, or 1/3 each) of the plain pair student with the listwise-10 and neighbour-batch + listwise-10 students (same 1k init; models are uploaded to Drive bitirme/models)
    'd_soup_pair_nblw10_a50': dict(model=f'{DRIVE_ROOT}/models/soup_pair_nblw10_a50'),
    'd_soup_pair_lw10_a50': dict(model=f'{DRIVE_ROOT}/models/soup_pair_lw10_a50'),
    'd_soup_pair_lw10_nblw10': dict(model=f'{DRIVE_ROOT}/models/soup_pair_lw10_nblw10'),
})
# The same students with a Normalize layer appended (suffix _norm). A folder saved by training is loaded as transformer
# + mean pooling only, so its vectors are not unit length. That matters for the classification and clustering tasks
# (a logistic regression / k-means is fitted on the raw vectors; sklearn's "lbfgs failed to converge" warnings in the
# log are a symptom), while retrieval, STS and pair classification use cosine and do not care. BGE-M3, mE5, Qwen3 and
# modernbert-tr-embed ship a Normalize layer; modernbert-tr-1k-embed does not.
PRESETS.update({f'{k}_norm': {**v, 'normalize': True} for k, v in list(PRESETS.items()) if k.startswith(('c_', 'd_'))})
print([k for k in PRESETS if k.startswith(('c_', 'd_')) and not k.endswith('_norm')])

In [ ]:
# plain protocol only: no query/passage prompts, so the `*_instr` presets (which reuse a plain run) are not used here
PRESETS = {k: v for k, v in PRESETS.items() if 'reuse_from' not in v and 'query_prompt' not in v and 'passage_prompt' not in v}
print(len(PRESETS), 'presets')

## 3. Run one evaluation

`TASKS = None` runs all 14 tasks; a list of names runs only those (results of a subset go to a separate `__subset` label).

In [ ]:
CHOICE = 'd_single_bge_m3'
TASKS = None   # e.g. ['STS22.v2', 'SIB200Classification']

from turkish_distill.eval.mizan import run_evaluation, summarize_dir, format_table

label = CHOICE if TASKS is None else f'{CHOICE}__subset'
label_dir = run_evaluation(output_dir=EVAL_DIR, label=label, task_names=TASKS, **PRESETS[CHOICE])
print(format_table({label: summarize_dir(label_dir)}))

Run several presets one after another. Each model runs in its OWN Python process, so one failing model does not stop the others; an interrupted preset resumes (finished tasks are skipped). Teachers first, then the main students.

In [ ]:
BATCH = ['bge_m3', 'modernbert_tr_embed', 'modernbert_tr_1k_embed', 'modernbert_tr_base_1k_raw', 'c_mnrl_s50_b128',
         'd_single_bge_m3', 'd_single_bge_m3_seed43', 'd_single_bge_m3_seed44', 'd_mixabl_mix70k', 'd_mixabl_cosmos70k',
         'd_single_qwen3_8b', 'd_single_me5', 'd_single_me5_center', 'd_single_cosmos_e5', 'd_two_qwen3_8b_bge_m3',
         # the teachers encoded the question texts with their query instruction (students still see raw text)
         'd_single_cosmos_e5_instr', 'd_single_qwen3_8b_instr',
         # trained on the 200K mix
         'd_scale_bge_mix200k', 'd_scale_8k_mix200k']

import time
from turkish_distill.eval.mizan import run_evaluation_isolated, summarize_dir, format_table

failed = {}
for name in BATCH:
    t0 = time.time()
    print(f'\n########## {name}')
    try:
        d = run_evaluation_isolated(label=name, output_dir=EVAL_DIR, **PRESETS[name])
        print(f'\n=== {name} done in {(time.time() - t0) / 60:.1f} min')
        print(format_table({name: summarize_dir(d)}))
    except Exception as exc:
        failed[name] = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'\n=== {name} FAILED after {(time.time() - t0) / 60:.1f} min: {failed[name]}')

print('\nfailed:', failed or 'none')

## 4. Compare finished runs

`Indep.(9)` is the development score (mean of the nine tasks TR-MTEB does not contain); `MTEB Score` is the leaderboard's score (mean of the task-type means); `Shared(5)` is the mean of the five tasks both benchmarks have.

In [ ]:
import glob
import pandas as pd
from turkish_distill.eval.mizan import format_table, summarize_dir

dirs = sorted(d for d in glob.glob(f'{EVAL_DIR}/*') if os.path.isdir(d) and not d.endswith('__subset'))
summaries = {os.path.basename(d): summarize_dir(d) for d in dirs}
print(format_table(summaries))

per_task = pd.DataFrame({k: v['per_task'] for k, v in summaries.items()}).round(2)
per_task